In [6]:
import nibabel as nib
import numpy as numpy
from scipy.ndimage import median_filter
from skimage import exposure
from skimage.restoration import denoise_bilateral
from scipy.ndimage import zoom
import vtk
from vtk.util import numpy_support
from vtk.util.numpy_support import vtk_to_numpy
import tomllib
import nrrd
import os
import sys

from dipy.align.imwarp import SymmetricDiffeomorphicRegistration
from dipy.align.metrics import CCMetric

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from image_processing_functions import load_nifti, write_vti, export_prep

case = "cut"
side = "Left"
sequence = "T1"
 
with open("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/config.toml", mode="rb") as f:
    config = tomllib.load(f)


base            = config["data"]["base"]
input_file      = base + config["data"]["nii_file"]
output_folder   = config["data"]["output_folder"]
segment_filename = config["data"]["segment_filename"]
segment_folder  = config["data"]["segment_folder"]

if case == "cut":
    input_file = input_file[:-4] + "_cropped_denoised.nii"

if side != "":
    input_file = os.path.join(os.path.dirname(input_file), side +"_" + os.path.basename(input_file))

mesh_offset = numpy.loadtxt(base + output_folder + "mesh_offset_"+sequence+".txt")


In [7]:
segment_voxel_origin = numpy.array((181,  98,  35))
new_voxel_origin = numpy.array((170, 50, 50))

In [8]:
mask, mask_header = nrrd.read(base + segment_folder + segment_filename)

origin_mask = numpy.array(mask_header["space origin"], dtype=float)
directions = [numpy.array(d, dtype=float) for d in mask_header["space directions"] if d is not None and not numpy.any(numpy.isnan(numpy.array(d, dtype=float)))]
directions = numpy.array(directions)

data, spacing, origin_cut, affine = load_nifti(input_file, False)
new_shape = data.shape



Original matrix shape =  (190, 270, 90)


In [9]:
diff = - new_voxel_origin + segment_voxel_origin
vti_origin = -mesh_offset - diff*spacing

In [ ]:

shift = segment_voxel_origin - new_voxel_origin       # -> (11, 48, -15)

new_mask = numpy.zeros(tuple(new_shape), dtype=mask.dtype)

dst_start = numpy.maximum(shift, 0)
dst_end = numpy.minimum(shift + numpy.array(mask.shape[:3]), new_shape)
src_start = dst_start - shift
src_end = dst_end - shift

new_mask[dst_start[0]:dst_end[0], dst_start[1]:dst_end[1], dst_start[2]:dst_end[2]] = \
    mask[src_start[0]:src_end[0], src_start[1]:src_end[1], src_start[2]:src_end[2]]

export_prep(new_mask)

write_vti(new_mask, spacing, vti_origin, base + "image/mask.vti")


Saved VTI file to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Tw-eye-analysis/image/mask.vti
